In [1]:
import torch as th
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
device = th.device('cuda' if th.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained('openai-community/gpt2').to(device)

In [ ]:
ids = tokenizer("hello", return_tensors='pt').input_ids.to(device)
logits = model(ids).logits  # [1, L, vocab]
print(logits.shape, model.config.vocab_size)

torch.Size([1, 4, 50257]) 50257


In [11]:
from lpe.utils import distributions as dists
from lpe.method_utils import *
from lpe.methods import MHIS, DA_MHIS, GPT2Adapter

In [13]:
adapter = GPT2Adapter(model, tokenizer, device=device)
behavior = distribution_registry["camel"](adapter.tokenizer, device=adapter.device)
orig_dists = behavior.input_dists(n_reps=N_REPS_DICT["camel"])

Repo card metadata block was not found. Setting CardData to empty.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 1024/1024 [00:05<00:00, 186.70it/s]


In [14]:
target = 123  # some token id
temp = 1.0    # tune as needed
est_mhis = MHIS(adapter, orig_dists, target, temp=temp, n_samples=2**12, burn_in=2**9, show_progress=False)
est_damh = DA_MHIS(adapter, orig_dists, target, temp=temp, n_samples=2**12, burn_in=2**9, show_progress=False)
print(est_mhis, est_damh)

/Users/ivanbetancourt25/Repos/low-probability-estimation/lpe/methods.py:282: SyntaxWarning: invalid escape sequence '\p'
  """


KeyboardInterrupt: 